# Whole-word masking for a padded batch

BERT chooses word pieces one at a time. When only `##bed` of `em ##bed ##ding ##s` is
hidden, the pieces on either side nearly spell it out, and the prediction teaches little.
The whole-word version of BERT's recipe, which Google released in 2019, chooses whole
words instead: once a word is chosen, every one of its pieces becomes a target. The
80/10/10 treatment is still decided piece by piece.

You will write the three steps for a padded batch of token IDs: number the words, choose
whole words up to a budget of pieces, and then corrupt the chosen pieces and build the
labels.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first. A piece whose text starts with `##` continues the word before it, so
the batch comes with a boolean array `continues` that is True at those pieces.

In [ ]:
import numpy as np

PAD_ID, CLS_ID, SEP_ID, MASK_ID = 0, 101, 102, 103
FIRST_PIECE, VOCAB = 1000, 30522    # ordinary pieces have IDs from 1000 up to 30521


def encode(rows, length):
    """Rows of piece strings -> padded input_ids, the continues array, and an id-to-piece map."""
    vocab = {"[PAD]": PAD_ID, "[CLS]": CLS_ID, "[SEP]": SEP_ID, "[MASK]": MASK_ID}
    ids = np.full((len(rows), length), PAD_ID)
    continues = np.zeros((len(rows), length), dtype=bool)
    for r, row in enumerate(rows):
        for t, piece in enumerate(["[CLS]"] + row.split() + ["[SEP]"]):
            vocab.setdefault(piece, FIRST_PIECE + len(vocab))
            ids[r, t] = vocab[piece]
            continues[r, t] = piece.startswith("##")
    return ids, continues, {i: p for p, i in vocab.items()}


rows = [
    "sea ice melt ##s early",
    "the ice ##berg ##s drift ##ed slowly south past the un ##char ##ted coast ##line of the north ##ern is ##lands",
]
input_ids, continues, id_to_piece = encode(rows, 24)
print(input_ids)
print(continues.astype(int))

## Part 1 (coding task): Number the words

Write `word_ids(input_ids, continues)`. It returns an integer array of the same shape. In
each row, number the words 0, 1, 2, ... from the left. Every piece of a word gets its
word's number, so a piece with `continues` True gets the number of the piece before it.
`[CLS]`, `[SEP]` and `[PAD]` get −1.

For the first row of the setup batch, `[CLS] sea ice melt ##s early [SEP]` followed by
`[PAD]`s, the answer starts `[-1, 0, 1, 2, 2, 3, -1, -1, ...]`.

In [ ]:
def word_ids(input_ids, continues):
    """Each piece's word number within its row, -1 at [CLS], [SEP] and [PAD]."""
    input_ids = np.asarray(input_ids)
    continues = np.asarray(continues, dtype=bool)
    # TODO
    raise NotImplementedError

In [ ]:
got = np.asarray(word_ids(input_ids, continues))
assert got.shape == input_ids.shape, f"shape {got.shape}: expected {input_ids.shape}, one number per position"
assert np.array_equal(got[0], [-1, 0, 1, 2, 2, 3] + [-1] * 18), f"row 0: got {got[0].tolist()}"
row1 = [-1, 0, 1, 1, 1, 2, 2, 3, 4, 5, 6, 7, 7, 7, 8, 8, 9, 10, 11, 11, 12, 12, -1, -1]
assert np.array_equal(got[1], row1), f"row 1: got {got[1].tolist()}; each row numbers its own words from 0"
ids2, cont2, _ = encode(["a b ##c", "d ##e ##f g ##h", "i"], 8)
expected = [[-1, 0, 1, 1, -1, -1, -1, -1], [-1, 0, 0, 0, 1, 1, -1, -1], [-1, 0, -1, -1, -1, -1, -1, -1]]
assert np.array_equal(word_ids(ids2, cont2), expected), f"got {np.asarray(word_ids(ids2, cont2)).tolist()}"
pair = np.array([[CLS_ID, 1500, 1501, SEP_ID, 1502, 1503, SEP_ID, PAD_ID]])
pair_cont = np.array([[False, False, True, False, False, True, False, False]])
assert np.array_equal(word_ids(pair, pair_cont), [[-1, 0, 0, -1, 1, 1, -1, -1]]), (
    "a [SEP] between two segments is not a word, and the numbering carries on after it"
)
print("Part 1 passed")

## Part 2 (coding task): Choose whole words

Write `choose_whole_words(input_ids, continues, rng, rate=0.15)`. It returns a boolean
array of the same shape that is True at every chosen piece. Treat each row on its own.

1. Let $n$ be the number of ordinary pieces in the row, everything except `[CLS]`, `[SEP]`
   and `[PAD]`. The budget is $k = \max(1, \operatorname{round}(\text{rate} \cdot n))$
   pieces.
2. Go through the row's words in a random order, given by `rng.permutation`. Choose a
   word if all of its pieces fit in what is left of the budget, and skip it if they do not.
3. Stop once $k$ pieces are chosen.

Use `word_ids` from Part 1. Python's `round` is fine here.

In [ ]:
def choose_whole_words(input_ids, continues, rng, rate=0.15):
    """True at every piece of the words chosen in each row, at most round(rate * n) pieces per row."""
    words = word_ids(input_ids, continues)
    # TODO
    raise NotImplementedError

In [ ]:
def single(n):
    """A row of n one-piece words."""
    return " ".join(f"w{i}" for i in range(n))


rows2 = [single(20), single(7), single(12), single(3), "long ##er ##st " + single(11), single(8)]
ids_b, cont_b, _ = encode(rows2, 22)
words_b = np.asarray(word_ids(ids_b, cont_b))
budgets = [3, 1, 2, 1, 2, 1]
for seed in range(300):
    c = np.asarray(choose_whole_words(ids_b, cont_b, np.random.default_rng(seed)))
    assert c.shape == ids_b.shape and c.dtype == bool, "return a boolean array shaped like input_ids"
    assert not c[words_b < 0].any(), "[CLS], [SEP] and [PAD] must never be chosen"
    counts = c.sum(axis=1)
    for r, (got, k) in enumerate(zip(counts, budgets)):
        n = int((words_b[r] >= 0).sum())
        assert got == k, (
            f"row {r}: its {n} ordinary pieces give a budget of max(1, round(0.15 * {n})) = {k}, "
            f"but the function chose {got} (seed {seed})"
        )
    for r in range(len(rows2)):
        for w in np.unique(words_b[r][words_b[r] >= 0]):
            part = c[r][words_b[r] == w]
            assert part.all() or not part.any(), f"row {r}, word {w}: choose all of a word's pieces or none"
assert not any(
    np.asarray(choose_whole_words(ids_b, cont_b, np.random.default_rng(s)))[4, 1:4].any() for s in range(50)
), "row 4: the three-piece word never fits a budget of 2"
ids_f, cont_f, _ = encode([single(20)], 22)
freq = np.mean([np.asarray(choose_whole_words(ids_f, cont_f, np.random.default_rng(s)))[0] for s in range(1000)], axis=0)
assert np.all(np.abs(freq[1:21] - 0.15) < 0.05), (
    f"each of the 20 words should be chosen about 15% of the time; got {np.round(freq[1:21], 2).tolist()}"
)
example = np.asarray(choose_whole_words(input_ids, continues, np.random.default_rng(4)))
print("chosen in the setup batch:")
for r in range(len(rows)):
    print("  ", " ".join(id_to_piece[i] for i, ch in zip(input_ids[r], example[r]) if ch))
print("Part 2 passed")

## Part 3 (coding task): Corrupt the chosen pieces and build the labels

Write `corrupt(input_ids, chosen, rng)`. It returns two new arrays, `inputs` and `labels`,
each shaped like `input_ids`, and leaves `input_ids` itself unchanged.

- `labels` holds the original piece at every chosen position and −100 everywhere else.
- `inputs` starts as a copy of `input_ids`. Each chosen piece, independently of the others,
  becomes `[MASK]` with probability 0.8, becomes a random piece drawn from
  `range(FIRST_PIECE, VOCAB)` with probability 0.1, and stays as it is with probability
  0.1.

In [ ]:
def corrupt(input_ids, chosen, rng):
    """BERT's 80/10/10 treatment of the chosen pieces, and the labels for the loss."""
    input_ids = np.asarray(input_ids)
    chosen = np.asarray(chosen, dtype=bool)
    # TODO
    raise NotImplementedError

In [ ]:
ids_c = np.random.default_rng(11).integers(FIRST_PIECE, VOCAB, size=(40, 500))
ids_c[:, 0], ids_c[:, -1] = CLS_ID, SEP_ID
chosen_c = np.zeros(ids_c.shape, dtype=bool)
chosen_c[:, 1:-1] = True
chosen_c[::2, 1:20] = False
before = ids_c.copy()
inputs_c, labels_c = corrupt(ids_c, chosen_c, np.random.default_rng(12))
inputs_c, labels_c = np.asarray(inputs_c), np.asarray(labels_c)
assert np.array_equal(ids_c, before), "input_ids was changed: work on a copy"
assert np.array_equal(labels_c, np.where(chosen_c, before, -100)), (
    "labels must be the original piece at chosen positions and -100 elsewhere"
    + ("; they contain [MASK], so they were read after corrupting" if (labels_c == MASK_ID).any() else "")
)
assert np.array_equal(inputs_c[~chosen_c], before[~chosen_c]), "pieces that were not chosen must stay as they are"
shown = inputs_c[chosen_c]
original = before[chosen_c]
masked = (shown == MASK_ID).mean()
kept = (shown == original).mean()
swapped = (shown != MASK_ID) & (shown != original)
print(f"of {chosen_c.sum()} chosen pieces: [MASK] {masked:.3f}, random {swapped.mean():.3f}, kept {kept:.3f}")
assert abs(masked - 0.8) < 0.015 and abs(swapped.mean() - 0.1) < 0.01 and abs(kept - 0.1) < 0.01, (
    "expected about 0.8 of the chosen pieces to show [MASK], 0.1 a random piece and 0.1 their own piece"
)
assert shown[swapped].min() >= FIRST_PIECE and shown[swapped].max() < VOCAB, (
    "random pieces must come from range(FIRST_PIECE, VOCAB), never a special token"
)
inputs_s, labels_s = corrupt(input_ids, choose_whole_words(input_ids, continues, np.random.default_rng(4)), np.random.default_rng(5))
for r in range(len(rows)):
    print("  ", " ".join(id_to_piece.get(int(i), "<random>") for i in np.asarray(inputs_s)[r] if i != PAD_ID))
print("Part 3 passed")

## Solution

### Part 1: Number the words

In [ ]:
def word_ids(input_ids, continues):
    """Each piece's word number within its row, -1 at [CLS], [SEP] and [PAD]."""
    input_ids = np.asarray(input_ids)
    continues = np.asarray(continues, dtype=bool)
    ordinary = ~np.isin(input_ids, [PAD_ID, CLS_ID, SEP_ID])
    starts = ordinary & ~continues                  # the first piece of every word
    numbers = np.cumsum(starts, axis=1) - 1         # words started so far in the row, minus one
    return np.where(ordinary, numbers, -1)

**Why it works.** A word starts at every ordinary piece that does not continue another one. Counting those
starts along the row with `cumsum` gives each position the number of words begun so far,
and subtracting 1 numbers them from 0. A continuation piece adds nothing to the count, so
it keeps the number of the word it belongs to. `axis=1` restarts the count in every row,
and `np.where` puts −1 back at the special tokens, which the count would otherwise
number like the piece before them.

### Part 2: Choose whole words

In [ ]:
def choose_whole_words(input_ids, continues, rng, rate=0.15):
    """True at every piece of the words chosen in each row, at most round(rate * n) pieces per row."""
    words = word_ids(input_ids, continues)
    chosen = np.zeros(words.shape, dtype=bool)
    for r, row in enumerate(words):
        n = int((row >= 0).sum())
        if n == 0:
            continue
        budget = max(1, round(rate * n))
        sizes = np.bincount(row[row >= 0])          # pieces in each word of the row
        used = 0
        for w in rng.permutation(len(sizes)):
            if used + sizes[w] <= budget:           # the whole word fits, so take it
                chosen[r] |= row == w
                used += sizes[w]
            if used == budget:
                break
    return chosen

**Why it works.** `np.bincount` over a row's word numbers gives the size of every word. A random permutation
of the word numbers puts each word first equally often, so every word has the same chance
of being chosen. A word is taken only when all its pieces fit, which keeps words whole and
the count at most $k$. Skipping a word that does not fit, instead of stopping there, lets
a later short word use the rest of the budget: in row 4 of the check, the three-piece word
comes first in one order out of twelve, and stopping at it would leave the row with no
target at all.

### Part 3: Corrupt the chosen pieces and build the labels

In [ ]:
def corrupt(input_ids, chosen, rng):
    """BERT's 80/10/10 treatment of the chosen pieces, and the labels for the loss."""
    input_ids = np.asarray(input_ids)
    chosen = np.asarray(chosen, dtype=bool)
    labels = np.where(chosen, input_ids, -100)
    inputs = input_ids.copy()
    roll = rng.random(input_ids.shape)              # one draw decides each piece's treatment
    inputs[chosen & (roll < 0.8)] = MASK_ID
    swap = chosen & (roll >= 0.8) & (roll < 0.9)
    inputs[swap] = rng.integers(FIRST_PIECE, VOCAB, swap.sum())
    return inputs, labels

**Why it works.** The labels are read from `input_ids` before anything is replaced, so every target is the
original piece, including the ones that end up showing `[MASK]`. `input_ids.copy()` keeps
the caller's array intact: `inputs = input_ids` would only give the same array a second
name, and the replacements would change the batch the caller still holds. A single
uniform draw per position, split at 0.8 and 0.9, gives each chosen piece one of the three
treatments, and only one, with the right probabilities. Two separate draws, one for `[MASK]` and one
for the swap, would let the second overwrite pieces the first had masked, and the shares
would come out near 0.72, 0.10 and 0.18.